# 01 | India viewing occasions

**Author: Chanakya**

Build a primary annual event-start ledger and a transparent ATP singles-slot sample. This is a timing-opportunity analysis, not an estimate of demand or an actual-start census.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/manifests/release.json').exists())
sys.path.insert(0, str(ROOT))
from src.analysis_common import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
rng = np.random.default_rng(CFG['seed'])
print('Offline inputs:', CFG['raw_release'], '| Author: Chanakya')
import src.analysis_common as shared
shared.ACTIVE_NOTEBOOK='01_viewing_calendar'
shared.ACTIVE_SOURCES=['atp_future_calendar_rendered_v2', 'brisbane_oop_20260105', 'brisbane_oop_20260110', 'brisbane_oop_20260111', 'doha_2026_official_op', 'dubai_2026_official_op', 'halle_2026_official_op', 'hongkong_2026_official_op', 'indian_wells_2026_official_op', 'madrid_2026_official_op', 'marrakech_2026_official_op', 'miami_2026_official_op', 'rome_2026_official_op', 'rotterdam_2026_official_op', 'v5_atp_media_guide_lta', 'winston_oop_pdf', 'winston_semifinal_oop']

## 1. Parse the primary annual guide
Preserve the entire original line and physical page. Multiweek continuation lines without draw counts are excluded. Dates inherited within a week are explicitly marked. The guide includes events outside the ATP rights package. Unresolved TBC entries remain visible.

In [ ]:
events=[];current=None
for page in [15,16]:
 for line in text('v5_atp_media_guide_lta',page).splitlines():
  date=re.search(r'\b(\d{2}) (JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)\b',line)
  if date:current=datetime.strptime(date.group(0)+' 2026','%d %b %Y').date().isoformat()
  cat=re.search(r'(ATP MASTERS 1000|ATP 500|ATP 250|GRAND SLAM|ATP FINALS|UNITED CUP|LAVER CUP)',line)
  if not cat or not re.search(r'(?:\d+|TEAMS)$',line.strip()):continue
  prefix=line[:cat.start()];city=re.sub(r'^\d+\s+','',prefix);city=re.sub(r'\b\d{2} (?:JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)\s*','',city);city=re.sub(r'[\d,]+\s*$','',city).strip()
  unknown='TBC' in line
  events.append(dict(event_id='G'+str(len(events)+1).zfill(3),city=city,tier=cat.group(0),start_date=None if unknown else current,date_inherited=not bool(date),source_line=line,pdf_page=page,source_id='v5_atp_media_guide_lta',rights_scope='ATP regular tour candidate' if cat.group(0) in ['ATP 250','ATP 500','ATP MASTERS 1000'] else 'Requires separate rights confirmation'))
events=pd.DataFrame(events);table(events,'atp_event_starts',True);display(events[['event_id','city','tier','start_date','date_inherited']])
display(table(events.groupby('tier').size().rename('event_rows').reset_index(),'01_calendar_tiers'))
# Exact dates for a bounded future journey are read from the saved current calendar capture.
future=[]
for line in text('atp_future_calendar_rendered_v2').splitlines():
 label,datepart=line.split(' | ');m=re.search(r'(\d+) - (\d+) (\w+), (\d+)',datepart)
 if m:a,b,month,year=m.groups();start=datetime.strptime(f'{a} {month} {year}','%d %B %Y');end=datetime.strptime(f'{b} {month} {year}','%d %B %Y')
 else:
  m=re.search(r'(\d+) (\w+) - (\d+) (\w+), (\d+)',datepart);a,ma,b,mb,year=m.groups();start=datetime.strptime(f'{a} {ma} {year}','%d %B %Y');end=datetime.strptime(f'{b} {mb} {year}','%d %B %Y')
 future.append(dict(event=label,start_date=start.date().isoformat(),end_date=end.date().isoformat(),source_id='atp_future_calendar_rendered_v2',regular_atp=not any(x in label for x in ['Davis','Laver'])))
future=pd.DataFrame(future);table(future,'atp_future_windows',True)
plt.figure(figsize=(11,4));d=events.dropna(subset=['start_date']).copy();d['month']=pd.to_datetime(d.start_date).dt.month
d.groupby(['month','tier']).size().unstack(fill_value=0).plot.bar(stacked=True,ax=plt.gca(),width=.85);plt.title('Published tournament starts across the year');plt.ylabel('Event starts');plt.xlabel('Month');plt.legend(fontsize=8,ncol=3);fig('01_event_calendar','Annual guide snapshot. TBC rows excluded from monthly chart. Rights scope differs by event.')

## 2. Convert a fixed 12-final sample and selected additional singles slots
Manual transcriptions are processed inputs with locators, not raw feeds. All clocks use IANA city timezones with date-aware daylight saving. Not-before times are lower bounds. Several source release stamps follow the planned match time, so this is retrospective schedule evidence, not proof of ex-ante campaign knowledge.

In [ ]:
slots=pd.DataFrame(json.loads((ROOT/'analysis_config/session_transcriptions.json').read_text()))
slots['start_utc']=[pd.Timestamp(f'{r.date_local} {r.time_local}',tz=r.timezone).tz_convert('UTC') for r in slots.itertuples()]
slots['start_ist']=pd.to_datetime(slots.start_utc,utc=True).dt.tz_convert('Asia/Kolkata')
slots['hour_ist']=slots.start_ist.dt.hour+slots.start_ist.dt.minute/60
slots['date_ist']=slots.start_ist.dt.date.astype(str);slots['weekday_ist']=slots.start_ist.dt.day_name()
slots['source_url']=[SOURCES[i]['url'] for i in slots.source_id]
table(slots,'atp_slots',True);display(table(slots[['event','round','date_local','time_local','timezone','start_ist','time_semantics','source_id']],'01_verified_slot_ledger'))
finals=slots[slots['round']=='Singles final'].copy()
plt.figure(figsize=(10,5));ax=plt.gca();ax.axvspan(18,23,color=COLORS[1],alpha=.13,label='Assumed 18:00–23:00 window')
for tier,g in finals.groupby('tier'):ax.scatter(g.hour_ist,g.event,s=90,label=tier,zorder=3)
ax.set(xlim=(0,24),xticks=range(0,25,3),xlabel='Scheduled / not-before start in IST',title='Tournament tier alone does not determine usable timing');ax.legend(fontsize=8,loc='lower left');fig('01_final_start_times','One final per preselected edition, n=12. Not-before is not actual start. Date rollover retained in table.')

## 3. Window and delay sensitivity
The preferred window is unknown. Test three declared windows and 0–120 minute delays. This shifts a start marker only, not a match-duration estimate. Report sample counts, not population confidence intervals.

In [ ]:
sens=[]
for start,end in CFG['viewing_windows']:
 for delay in CFG['scheduled_delay_minutes']:
  hours=(finals.hour_ist+delay/60)%24
  for event,tier,ok in zip(finals.event,finals.tier,(hours>=start)&(hours<end)):
   sens.append(dict(event=event,tier=tier,window=f'{start}:00–{end}:00',delay_minutes=delay,starts_in_window=bool(ok)))
sens=pd.DataFrame(sens);table(sens,'01_window_delay_sensitivity')
rob=sens.groupby('event').starts_in_window.agg(['sum','count','mean']).reset_index().rename(columns={'mean':'fraction_of_design_cells'})
display(table(rob,'01_timing_robustness'))
piv=sens[sens.delay_minutes==0].pivot(index='event',columns='window',values='starts_in_window').astype(int)
plt.figure(figsize=(8,5));plt.imshow(piv,aspect='auto',cmap='YlGnBu',vmin=0,vmax=1);plt.xticks(range(len(piv.columns)),piv.columns);plt.yticks(range(len(piv)),piv.index);plt.title('Does the final start fit the assumed window?');cb=plt.colorbar(ticks=[0,1]);cb.ax.set_yticklabels(['Outside','Inside']);fig('01_window_sensitivity','Binary scheduled-start fit. Unknown preference, not measured audience propensity.')
# Coverage is sparse by construction. A blank week is missing evidence, not zero tennis.
heat=np.full((53,24),np.nan)
for r in slots.itertuples():
 w=int(r.start_ist.isocalendar().week)-1;h=int(r.hour_ist);heat[w,h]=0 if np.isnan(heat[w,h]) else heat[w,h];heat[w,h]+=1
plt.figure(figsize=(11,4));plt.imshow(heat.T,aspect='auto',origin='lower',cmap='viridis');plt.colorbar(label='Known selected singles starts');plt.xlabel('ISO week index (week 1 at zero)');plt.ylabel('IST hour');plt.title('Coverage-aware sampled timing heatmap');fig('01_sampled_week_hour','White cells are unobserved, not absence of tennis. Use the final comparison for balanced tier analysis.')
check('01_calendar',{'twelve_distinct_finals':len(finals)==12 and finals.event.nunique()==12,'unique_slots':slots.slot_id.is_unique,'all_times_timezone_aware':all(x.tzinfo is not None for x in slots.start_utc),'dubai_ist_2030':float(finals.set_index('event').loc['Dubai','hour_ist'])==20.5,'indian_wells_next_day':finals.set_index('event').loc['Indian Wells','date_ist']=='2026-03-16'})
report('01_calendar_findings',f'{len(events)} guide rows retained, including unresolved dates and events outside the regular ATP package. {len(slots)} selected singles slots across 12 editions. Timing robustness is a design-grid fraction, not a probability. Source locators and full date rollover are preserved. Use clear included-access messaging and a live/replay choice before assuming a higher tier deserves more spend.')

## Decision passed forward
Target declared viewing occasions rather than a blanket “Asian events are prime-time” rule. The next notebook tests conflicts. Do not sell a player-specific live promise until participation and a usable slot are confirmed.

## Source references
These IDs resolve to the preserved bodies, URLs and capture timestamps. Derived tables also retain row-level source IDs where applicable. Case inputs refer to the supplied brief, physical PDF pages 9–14. Review source files resolve through the review collection log. Scenario parameters are in analysis_config.

In [ ]:
references=source_table(['atp_future_calendar_rendered_v2', 'brisbane_oop_20260105', 'brisbane_oop_20260110', 'brisbane_oop_20260111', 'doha_2026_official_op', 'dubai_2026_official_op', 'halle_2026_official_op', 'hongkong_2026_official_op', 'indian_wells_2026_official_op', 'madrid_2026_official_op', 'marrakech_2026_official_op', 'miami_2026_official_op', 'rome_2026_official_op', 'rotterdam_2026_official_op', 'v5_atp_media_guide_lta', 'winston_oop_pdf', 'winston_semifinal_oop'])
display(table(references,'01_source_references'))